# 7.6 — The primitive underneath: LangGraph's `interrupt()`

This notebook builds a small **custom LangGraph graph by hand** (no
`create_agent`) to show the low-level primitive that
`HumanInTheLoopMiddleware` wraps. Use this pattern when:

- You need to pause somewhere that isn't a tool call (e.g. mid-plan,
  asking the human to choose a direction)
- You're building a custom multi-agent graph, not a single
  `create_agent` loop
- You need full control over exactly what state is shown / edited at
  the pause point


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


## 1. Define graph state and nodes

The graph: draft a plan → **pause for human review** → finalize.
This is the kind of pause `HumanInTheLoopMiddleware` *can't* express
directly, because it isn't gating a tool call — it's gating a
free-form plan.


In [ ]:
class PlanState(TypedDict):
    topic: str
    draft_plan: str
    approved_plan: str


def draft(state: PlanState) -> PlanState:
    resp = llm.invoke(
        f"Draft a 3-step migration plan for: {state['topic']}. Keep it short."
    )
    return {"draft_plan": resp.content}


def human_review(state: PlanState) -> PlanState:
    # interrupt() pauses the graph here and persists state via the checkpointer.
    # Execution resumes with whatever value is passed to Command(resume=...).
    decision = interrupt({
        "question": "Approve this plan, or provide edits?",
        "draft_plan": state["draft_plan"],
    })
    return {"approved_plan": decision.get("edited_plan", state["draft_plan"])}


def finalize(state: PlanState) -> PlanState:
    print("FINAL PLAN:\n", state["approved_plan"])
    return {}


## 2. Wire the graph together with a checkpointer

In [ ]:
graph = StateGraph(PlanState)
graph.add_node("draft", draft)
graph.add_node("human_review", human_review)
graph.add_node("finalize", finalize)

graph.add_edge(START, "draft")
graph.add_edge("draft", "human_review")
graph.add_edge("human_review", "finalize")
graph.add_edge("finalize", END)

checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer)


## 3. Run until the interrupt

In [ ]:
config = {"configurable": {"thread_id": "plan-thread-1"}}

result = app.invoke({"topic": "Netezza to Databricks Lakehouse"}, config)
print("Interrupted:", "__interrupt__" in result)
result.get("__interrupt__")


## 4. Resume — either approve as-is or edit

Because this is raw `interrupt()`, the shape of the resume payload is
entirely up to you — unlike the middleware's fixed
approve/edit/reject/respond schema.


In [ ]:
# Approve as drafted:
final = app.invoke(Command(resume={}), config)


In [ ]:
# ...or, starting a fresh thread, edit the plan before continuing:
config2 = {"configurable": {"thread_id": "plan-thread-2"}}
app.invoke({"topic": "Netezza to Databricks Lakehouse"}, config2)

final_edited = app.invoke(
    Command(resume={"edited_plan": "1. Freeze writes 2. Dual-run 2 weeks 3. Cutover on Sunday"}),
    config2,
)


## Recap: where each tool lives

| Need | Use |
|---|---|
| Approve/edit/reject a **tool call** inside `create_agent` | `HumanInTheLoopMiddleware` (notebook 7.5, Part 4) |
| Pause at an **arbitrary point** for arbitrary human input | Raw `interrupt()` + `Command(resume=...)` (this notebook) |
| Persist state across the pause, either way | LangGraph's checkpointer (`InMemorySaver` here; swap for a durable one — e.g. Postgres — in production) |

Both paths ultimately rely on the **same LangGraph mechanism**:
`interrupt()` pauses, the checkpointer persists, `Command(resume=...)`
continues. The middleware is a convenience layer for the single most
common shape of that pattern.
